In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

FILE_OLD = "data/raw/HS861M 1990-2009.xlsx"
FILE_NEW = "data/raw/HS861M 2010-.xlsx"

# EIA-861M Monthly Electricity Data Cleaning

Cleans and combines the two monthly EIA-861M files into a single CSV

In [ ]:
# Both files have a 3-row merged header; skip to the third row for column names
def load_eiam(path):
    df = pd.read_excel(path, sheet_name=0, header=2)
    return df

old = load_eiam(FILE_OLD)
new = load_eiam(FILE_NEW)

print("Old shape:", old.shape)
print("New shape:", new.shape)
old.head(3)

# Rename the first four columns consistently across both files
def rename_base_cols(df):
    cols = list(df.columns)
    cols[:4] = ["year", "month", "state", "data_status"]
    df.columns = cols
    return df

old = rename_base_cols(old)
new = rename_base_cols(new)

df = pd.concat([old, new], ignore_index=True)
df["year"] = pd.to_numeric(df["year"], errors="coerce")
df = df.dropna(subset=["year"])
df["year"] = df["year"].astype(int)

print("Combined shape:", df.shape)
print("Year range:", df['year'].min(), "–", df['year'].max())

# The EIA file is wide: Revenue / Sales / Customers / Price for each sector
# Column positions (0-indexed): residential price = col 7, commercial = 11, industrial = 15
# We keep year, month, state, data_status + all four price columns
price_cols = {"year": df.columns[0],
              "month": df.columns[1],
              "state": df.columns[2],
              "data_status": df.columns[3],
              "res_price": df.columns[7],
              "com_price": df.columns[11],
              "ind_price": df.columns[15],
              "total_price": df.columns[-1]}

df = df[list(price_cols.values())].copy()
df.columns = list(price_cols.keys())
#df.head()

price_col_names = ["res_price", "com_price", "ind_price", "total_price"]

df[price_col_names] = df[price_col_names].replace(".", np.nan)
df[price_col_names] = df[price_col_names].apply(pd.to_numeric, errors="coerce")

print("Missing values per column:")
print(df[price_col_names].isna().sum())

STATES = ["VA", "MD", "NC", "TX", "CA", "US"]

df_states = df[df["state"].isin(STATES)].copy()

print("Filtered shape:", df_states.shape)
print("States included:", sorted(df_states["state"].unique()))

df_states["date"] = pd.to_datetime(
    df_states[["year", "month"]].assign(day=1)
)

df_states = df_states.sort_values(["state", "date"]).reset_index(drop=True)
#df_states.head()



Old shape: (12241, 28)
New shape: (10201, 24)
Combined shape: (22440, 28)
Year range: 1990 – 2026
Missing values per column:
res_price          0
com_price         13
ind_price         13
total_price    10200
dtype: int64
Filtered shape: (2200, 8)
States included: ['CA', 'MD', 'NC', 'TX', 'VA']
year                    int64
month                 float64
state                     str
data_status               str
res_price             float64
com_price             float64
ind_price             float64
total_price           float64
date           datetime64[us]
dtype: object

              year        month    res_price    com_price    ind_price  \
count  2200.000000  2200.000000  2200.000000  2200.000000  2200.000000   
mean   2007.836364     6.463636    11.639805     9.335982     7.059077   
min    1990.000000     1.000000     5.740000     4.300000     2.620000   
25%    1999.000000     3.000000     8.320000     6.710000     4.740000   
50%    2008.000000     6.000000    11.110000     